Otsu's Thresholding & Adaptive Thresholding

Welcome to this interactive tutorial. In computer vision, thresholding is a technique used to convert a grayscale image into a binary image. It is heavily used to separate the foreground from the background.

In this notebook, we will see how Otsu's Method works automatically with example and explication.

In [ ]:
Step 1: Environment Setup and Image Loading

First we need to import the libraries. We will use cv2 (OpenCV) to load our test 
image and directly convert it in grayscale in a single line of code, whe will also need matplotlib to visualize our intermediate results in this notebook

Before running the code cell below, you must ensure that you have an image file named exactly test_image.jpg placed in the exact same folder as this Notebook.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load the image directly in grayscale in a single line
image_path = 'test_image.jpg'
gray_image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

if gray_image is None:
    print("Error: Could not load image. Check the file path.")
else:
    plt.figure(figsize=(6, 4))
    plt.imshow(gray_image, cmap='gray')
    plt.title("Step 1: Original Image (Loaded directly in Grayscale)")
    plt.axis('off')
    plt.show()

Step 2: The Bimodal Assumption and Histogram Analysis

Before applying Otsu's method, we need to understand how it makes its decision. Otsu's algorithm does not look at the image as a 2D spatial grid. instead, it looks at the histogram of pixel intensities.

The algorithm assumes that the image contains two main classes of pixels: the foreground and the background. Ideally, this creates a graph with two distinct mountains or peaks, separated by a deep valley.

To find the perfect cut-off point, Otsu's method iterates through all possible threshold values (from 0 to 255). For each value, it calculates the within-class variance. The optimal threshold is the one that minimizes this variance, meaning it ensures the pixels inside the "background" group are as similar to each other as possible, and the same goes for the "foreground" group.

Let's plot the histogram of our loaded image to see if it respects this bimodal assumption!

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Flatten the 2D image matrix into a 1D list
# Ensure gray_image is a numpy array to avoid attribute errors
pixels = np.asarray(gray_image).ravel()
# Create and plot the histogram
plt.figure(figsize=(8, 5))
# Use 256 bins for the 256 possible intensity values (0 to 255)
plt.hist(pixels, bins=256, range=(0, 256), color='gray', alpha=0.8)
plt.title("Step 2: Histogram of Pixel Intensities")
plt.xlabel("Pixel Intensity (0 = Pure Black, 255 = Pure White)")
plt.ylabel("Number of Pixels (Frequency)")
plt.grid(axis='y', alpha=0.3)
# Display the graph
plt.show()

Step 3: Applying Otsu's Method and Binarization

Now that we have analyzed the histogram and verified its bimodal nature, we can let the algorithm do the work. Instead of manually guessing a threshold value like 127, we will use Otsu's method to automatically find that perfect valley between our two pixel classes.

Applying this calculated threshold converts our grayscale image into a strict binary format. Every pixel is forced to become either completely black or completely white. This process creates a mask, cleanly separating our objects from the background.

In OpenCV, this is done using the standard threshold function. By combining the normal binary flag with Otsu's flag, we tell the library to ignore whatever manual threshold number we input, calculate the optimal value based on the variance, and apply it directly to the image.

Let's run the algorithm on our image and visualize the segmented result to see what threshold value the computer automatically chose.

In [ ]:
import cv2
import matplotlib.pyplot as plt

# Apply binary thresholding combined with Otsu's method
# We pass 0 as the threshold value because the THRESH_OTSU flag will calculate and override it
optimal_threshold, binary_image = cv2.threshold(gray_image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Print the value found by the algorithm
print(f"Otsu's algorithm calculated the optimal threshold as: {optimal_threshold}")

# Create a figure to compare the before and after
plt.figure(figsize=(10, 5))

# Plot the original grayscale image
plt.subplot(1, 2, 1)
plt.imshow(gray_image, cmap='gray')
plt.title("Grayscale Image")
plt.axis('off')

# Plot the new binary image
plt.subplot(1, 2, 2)
plt.imshow(binary_image, cmap='gray')
plt.title(f"Binary Image (Otsu Threshold = {optimal_threshold})")
plt.axis('off')

# Display the graphics
plt.show()

Step 4: Overcoming Lighting Issues with Adaptive Thresholding

While Otsu's method is highly effective for images with uniform lighting, it relies on calculating a single global threshold applied to the entire picture simultaneously. If an image contains heavy shadows, gradients, or uneven illumination, a global threshold will often fail. It might completely erase critical details hidden in the darker areas while overexposing the brightly lit sections. To solve this specific problem, we must transition from a global approach to a local one using Adaptive Thresholding.

Instead of determining one universal cut-off value, adaptive thresholding divides the image into smaller, localized grids or neighborhoods. The algorithm then calculates a unique, independent threshold for each specific local area based on the surrounding pixels. By doing this, every part of the image is evaluated relative to its own specific lighting conditions. A dark pixel in a shadowed corner is compared only to other shadowed pixels, ensuring that structural details are preserved regardless of the overall lighting gradient.

In OpenCV, we apply this technique by defining a block size, which represents the dimensions of the local neighborhood, and a constant value to fine-tune the sensitivity. We will use a Gaussian method, where the threshold value is the weighted sum of neighborhood values, providing a smoother and more natural result than a simple mathematical mean. Let us apply this adaptive technique and compare it directly alongside our previous global Otsu result to observe the difference in detail retention.